In [1]:
import os
import time
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
from torchvision import transforms, models

# Use all available CPU threads
torch.set_num_threads(os.cpu_count() or 1)



In [2]:
data_path = "../input/cassava-leaf-disease-classification/"
train_path = "../input/cassava-leaf-disease-classification/train_images/"
test_path = "../input/cassava-leaf-disease-classification/test_images/"




In [3]:
class ResNet18Cassava(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Linear(self.model.fc.in_features, n_classes)

    def forward(self, x):
        return self.model(x)




In [4]:
# Choose device: GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cassava_model = ResNet18Cassava(n_classes=5).to(device)

if hasattr(torch, "compile"):
    cassava_model = torch.compile(cassava_model)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cassava_model.parameters(), lr=1.5e-05)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 114MB/s]


In [5]:
class TrainSet(Dataset):
    """Cassava Disease Training Dataset"""

    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["image_id"]
        label = int(row["label"])
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label




In [6]:
train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

full_train_dataset = TrainSet(
    csv_path=os.path.join(data_path, "train.csv"),
    img_dir=train_path,
    transform=train_transform,
)

train_len = int(0.9 * len(full_train_dataset))
val_len = len(full_train_dataset) - train_len
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_len, val_len],
    generator=torch.Generator().manual_seed(42),
)

# Increased batch sizes and enabled persistent workers for faster loading
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=256,  # larger batch reduces iteration count
    shuffle=True,
    num_workers=4,
    pin_memory=True,  # speeds up transfer when using GPU
    persistent_workers=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=512,  # larger batch for validation
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)



In [7]:
num_epochs = 2
for epoch in range(1, num_epochs + 1):
    cassava_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = cassava_model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    cassava_model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = cassava_model(imgs)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch}/{num_epochs} - "
        f"Train loss: {epoch_loss:.4f}, Train acc: {epoch_acc:.4f}, "
        f"Val acc: {val_acc:.4f}"
    )




/usr/local/lib/python3.11/dist-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Epoch 1/2 - Train loss: 1.1985, Train acc: 0.5228, Val acc: 0.6946
Epoch 2/2 - Train loss: 0.7823, Train acc: 0.7361, Val acc: 0.7549


In [8]:
class TestSet2(Dataset):
    """Cassava Disease Test Dataset"""

    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(self.test_dir) if f.lower().endswith(".jpg")
        ]
        print(f"Cassava Disease Test Dataset Length = {len(self.image_files)}")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, img_name




In [9]:
test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

testset = TestSet2(test_dir=test_path, transform=test_transform)
test_loader = DataLoader(
    dataset=testset,
    batch_size=256,  # larger batch for faster inference
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)



Cassava Disease Test Dataset Length = 2676


In [10]:
col_names = ["image_id", "label"]
rows = []

tic = time.time()
cassava_model.eval()
with torch.no_grad():
    for X_test, names in test_loader:
        X_test = X_test.to(device)
        logits = cassava_model(X_test)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        for img_name, pred in zip(names, preds):
            rows.append({"image_id": img_name, "label": int(pred)})
toc = time.time() - tic
print("Inference time:", toc)

submission_df = pd.DataFrame(rows, columns=col_names)
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print(submission_df.head())

Inference time: 5.105259418487549
Submission saved to submission.csv
         image_id  label
0  2574872277.jpg      3
1  1449210447.jpg      3
2  1269550303.jpg      4
3  1411799307.jpg      3
4  2474336811.jpg      3
